## **Evaluation of Extensive Cleaning Method**

In [1]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

In [2]:
spark = sparknlp.start()

26/04/14 16:14:55 WARN Utils: Your hostname, MacBook-Pro-4.local resolves to a loopback address: 127.0.0.1; using 10.0.0.127 instead (on interface en0)
26/04/14 16:14:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/leena/Desktop/tweet-sentiment-analysis/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/leena/.ivy2/cache
The jars for the packages stored in: /Users/leena/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0fca8274-8eae-4327-8d56-6c1665a91d49;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in centra

26/04/14 16:14:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/14 16:14:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Load Data

In [3]:
CLEAN_DATA_PATH = '../cleaned_data/nlp_data.csv'

In [4]:
df = spark.read.option("header", True).csv(CLEAN_DATA_PATH)
df.show(5)
print("Number of rows:", df.count())

+----+-----------+------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|
+----+-----------+------------+--------------------+
|2401|Borderlands|    Positive|im get borderland...|
|2401|Borderlands|    Positive|    come border kill|
|2401|Borderlands|    Positive|im get borderland...|
|2401|Borderlands|    Positive|im come borderlan...|
|2401|Borderlands|    Positive|im get borderland...|
+----+-----------+------------+--------------------+
only showing top 5 rows

Number of rows: 74682


### Drop Nulls and Empty Values

In [5]:
df = df.dropna(subset=["cleaned_text"])
print("Number of rows after dropping nulls:", df.count())

Number of rows after dropping nulls: 72389


In [ ]:
from pyspark.sql.functions import trim

df = df.filter(trim(col("cleaned_text")) != "")
print("Number of rows after dropping empty strings:", df.count())

Number of rows after dropping empty strings: 72389


### Drop Rows with "Irrelevant" OG Sentiment
These must be dropped as the model can only classify "positive", "neutral", or "negative". 

In [ ]:
df = df.filter(col("og_sentiment") != "Irrelevant")
print("Number of rows after dropping irrelevant rows:", df.count())

Number of rows after dropping irrelevant rows: 59755


### Run Model

In [8]:
document_assembler = DocumentAssembler() \
    .setInputCol('cleaned_text') \
    .setOutputCol('document')

tokenizer = Tokenizer() \
    .setInputCols(['document']) \
    .setOutputCol('token')

classifier = XlmRoBertaForSequenceClassification.pretrained('twitter_xlm_roberta_base_sentiment')\
  .setInputCols(["document",'token'])\
  .setOutputCol("pred_sentiment")

pipeline = Pipeline(stages=[
    document_assembler, 
    tokenizer,
    classifier    
])

model = pipeline.fit(df)
result = model.transform(df)

twitter_xlm_roberta_base_sentiment download started this may take some time.
Approximate size to download 993.6 MB
[ | ]26/04/14 16:15:14 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/04/14 16:15:14 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
twitter_xlm_roberta_base_sentiment download started this may take some time.
Approximate size to download 993.6 MB
[ | ]Download done! Loading the resource.
[ — ]

[OK!]


In [9]:
result.show()

Using CPUs
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 23...|[{token, 0, 1, im...|[{category, 0, 23...|
|2401|Borderlands|    Positive|    come border kill|[{document, 0, 15...|[{token, 0, 3, co...|[{category, 0, 15...|
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 21...|[{token, 0, 1, im...|[{category, 0, 21...|
|2401|Borderlands|    Positive|im come borderlan...|[{document, 0, 24...|[{token, 0, 1, im...|[{category, 0, 24...|
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 23...|[{token, 0, 1, im...|[{category, 0, 23...|
|2401|Borderlands|    Positive|im get borderland...|[{documen

In [ ]:
# View the actual labels from the pred_sentiment column
result.select("og_sentiment", "cleaned_text", "pred_sentiment.result").show()

+------------+--------------------+----------+
|og_sentiment|        cleaned_text|    result|
+------------+--------------------+----------+
|    Positive|im get borderland...| [neutral]|
|    Positive|    come border kill| [neutral]|
|    Positive|im get borderland...| [neutral]|
|    Positive|im come borderlan...| [neutral]|
|    Positive|im get borderland...| [neutral]|
|    Positive|im get borderland...| [neutral]|
|    Positive|spend hour make s...|[positive]|
|    Positive|spend couple hour...|[positive]|
|    Positive|spend hour someth...|[positive]|
|    Positive|spend hour make s...|[positive]|
|    Positive|spend hour make s...|[positive]|
|     Neutral|RockHard La Varlo...|[positive]|
|     Neutral|RockHard La Varlo...|[positive]|
|     Neutral|RockHard La Varlo...|[positive]|
|     Neutral|RockHard La Vita ...| [neutral]|
|     Neutral|Live Rock Hard mu...|[positive]|
|     Neutral|IHard like RARE L...| [neutral]|
|    Positive|first borderland ...| [neutral]|
|    Positive

In [ ]:
# Create column with the predicted labels
result = result.withColumn("pred_label",col("pred_sentiment")[0].result)
result.show()

+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|pred_label|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 23...|[{token, 0, 1, im...|[{category, 0, 23...|   neutral|
|2401|Borderlands|    Positive|    come border kill|[{document, 0, 15...|[{token, 0, 3, co...|[{category, 0, 15...|   neutral|
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 21...|[{token, 0, 1, im...|[{category, 0, 21...|   neutral|
|2401|Borderlands|    Positive|im come borderlan...|[{document, 0, 24...|[{token, 0, 1, im...|[{category, 0, 24...|   neutral|
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 23...|[{token, 0, 1, im...|[{category, 0, 23

### Normalize Label Columns

In [15]:
from pyspark.sql.functions import lower

result = result.withColumn("og_label", lower(col("og_sentiment"))) \
               .withColumn("pred_label", lower(col("pred_label")))

result.show()

+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+--------+
|  id|      topic|og_sentiment|        cleaned_text|            document|               token|      pred_sentiment|pred_label|og_label|
+----+-----------+------------+--------------------+--------------------+--------------------+--------------------+----------+--------+
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 23...|[{token, 0, 1, im...|[{category, 0, 23...|   neutral|positive|
|2401|Borderlands|    Positive|    come border kill|[{document, 0, 15...|[{token, 0, 3, co...|[{category, 0, 15...|   neutral|positive|
|2401|Borderlands|    Positive|im get borderland...|[{document, 0, 21...|[{token, 0, 1, im...|[{category, 0, 21...|   neutral|positive|
|2401|Borderlands|    Positive|im come borderlan...|[{document, 0, 24...|[{token, 0, 1, im...|[{category, 0, 24...|   neutral|positive|
|2401|Borderlands|    Positive|im get borderland

### Calculate Accuracy

In [16]:
eval_df = result.select("og_label", "pred_label").toPandas()
eval_df

,og_label,pred_label
0,positive,neutral
1,positive,neutral
2,positive,neutral
3,positive,neutral
4,positive,neutral
...,...,...
59750,positive,neutral
59751,positive,negative
59752,positive,negative
59753,positive,neutral


In [18]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(eval_df["og_label"], eval_df["pred_label"])
print("Accuracy:", accuracy)

Accuracy: 0.3397372604802945


### Calculate F1 Scores

In [19]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(eval_df["og_label"], eval_df["pred_label"], average="macro")
print("Macro F1:", macro_f1)

weighted_f1 = f1_score(eval_df["og_label"], eval_df["pred_label"], average="weighted")
print("Weighted F1:", weighted_f1)

Macro F1: 0.30731640978437413
Weighted F1: 0.29893817581644144


### Classification Report

In [21]:
from sklearn.metrics import classification_report

print(classification_report(eval_df["og_label"], eval_df["pred_label"]))

              precision    recall  f1-score   support

    negative       0.59      0.08      0.13     21916
     neutral       0.29      0.56      0.39     17654
    positive       0.38      0.43      0.40     20185

    accuracy                           0.34     59755
   macro avg       0.42      0.36      0.31     59755
weighted avg       0.43      0.34      0.30     59755

